<a href="https://colab.research.google.com/github/abdulahmd/abdulahmd.github.io/blob/main/DroneControlPresentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Imports & Sensor Setup

In [ ]:
import serial
import time
import numpy as np
import matplotlib.pyplot as plt
from dronekit import connect, VehicleMode, LocationGlobalRelative
import random
import math

vehicle = connect('/dev/ttyAMA0', wait_ready=True, baud=57600)

def get_co2_level():
    try:
        with serial.Serial('/dev/ttyS0', baudrate=9600, timeout=1) as ser:
            ser.write(b'\xFF\x01\x86\x00\x00\x00\x00\x00\x79')
            response = ser.read(9)
            if response[0] == 0xFF and response[1] == 0x86:
                high = response[2]
                low = response[3]
                co2 = (high << 8) + low
                return co2
    except Exception as e:
        print("CO2 sensor error:", e)
    return None


### Node Setup for RRT* Algorithm (Data Structure)

In [ ]:
class Node:
    def __init__(self, position):
        self.position = position
        self.parent = None
        self.cost = 0


### RRT* Path Algorithm

In [ ]:
class RRTStar:
    def __init__(self, start, goal, obstacle_list, map_size, max_iter=300, step_size=1.0, search_radius=2.0):
        self.start = Node(start)
        self.goal = Node(goal)
        self.nodes = [self.start]
        self.obstacles = obstacle_list
        self.map_size = map_size
        self.max_iter = max_iter
        self.step_size = step_size
        self.search_radius = search_radius

    def get_random_point(self):
        return (random.uniform(0, self.map_size[0]), random.uniform(0, self.map_size[1]))

    def get_nearest_node(self, point):
        return min(self.nodes, key=lambda node: np.linalg.norm(np.array(node.position) - np.array(point)))

    def is_collision_free(self, p1, p2):
        for (ox, oy, r) in self.obstacles:
            dx, dy = np.array(p2) - np.array(p1)
            a = dx * dx + dy * dy
            b = 2 * (dx * (p1[0] - ox) + dy * (p1[1] - oy))
            c = ox**2 + oy**2 + p1[0]**2 + p1[1]**2 - 2 * (ox * p1[0] + oy * p1[1]) - r**2
            if b**2 - 4 * a * c >= 0:
                return False
        return True

    def plan(self):
        for _ in range(self.max_iter):
            rand_point = self.get_random_point()
            nearest_node = self.get_nearest_node(rand_point)
            direction = np.array(rand_point) - np.array(nearest_node.position)
            length = np.linalg.norm(direction)
            if length == 0:
                continue
            new_point = tuple(np.array(nearest_node.position) + (direction / length) * self.step_size)
            if not self.is_collision_free(nearest_node.position, new_point):
                continue
            new_node = Node(new_point)
            near_nodes = [node for node in self.nodes if np.linalg.norm(np.array(node.position) - np.array(new_point)) < self.search_radius]
            best_node = nearest_node
            min_cost = nearest_node.cost + np.linalg.norm(np.array(new_point) - np.array(nearest_node.position))

            for node in near_nodes:
                if self.is_collision_free(node.position, new_point):
                    cost = node.cost + np.linalg.norm(np.array(new_point) - np.array(node.position))
                    if cost < min_cost:
                        best_node = node
                        min_cost = cost

            new_node.parent = best_node
            new_node.cost = min_cost
            self.nodes.append(new_node)

            # Rewiring
            for node in near_nodes:
                if self.is_collision_free(new_node.position, node.position):
                    cost = new_node.cost + np.linalg.norm(np.array(new_node.position) - np.array(node.position))
                    if cost < node.cost:
                        node.parent = new_node
                        node.cost = cost

        # Backtrace final path
        last_node = min(self.nodes, key=lambda node: np.linalg.norm(np.array(node.position) - np.array(self.goal.position)))
        path = []
        while last_node is not None:
            path.append(last_node.position)
            last_node = last_node.parent
        return path[::-1]


### Drone Flight Telemetry

In [ ]:
def goto(position, altitude=2.0):
    print(f"Going to: {position}")
    loc = LocationGlobalRelative(position[0], position[1], altitude)
    vehicle.simple_goto(loc)
    time.sleep(5)


### Drone Flight + CO2 data gathering

In [ ]:
if __name__ == "__main__":
    start = (0, 0)
    goal = (10, 10)
    obstacles = [(5, 5, 1.5), (3, 6, 1.0)]
    map_size = (15, 15)

    rrt_star = RRTStar(start, goal, obstacles, map_size)
    path = rrt_star.plan()

    print("Arming and taking off...")
    vehicle.mode = VehicleMode("GUIDED")
    vehicle.armed = True
    while not vehicle.armed:
        time.sleep(1)
    vehicle.simple_takeoff(2.0)

    time.sleep(5)

    for waypoint in path:
        goto(waypoint)
        co2 = get_co2_level()
        if co2 is not None:
            print(f"CO₂ level at {waypoint}: {co2} ppm")
        else:
            print("Failed to read CO₂ sensor")

    print("Mission complete")
    vehicle.mode = VehicleMode("LAND")
    vehicle.close()
